In [3]:
pip install mcap-ros2-support opencv-python rosbags

     ---------------------------------------- 44.0/44.0 MB 8.3 MB/s eta 0:00:00
     -------------------------------------- 144.9/144.9 kB 9.0 MB/s eta 0:00:00
     ---------------------------------------- 3.3/3.3 MB 9.0 MB/s eta 0:00:00
     ---------------------------------------- 99.5/99.5 kB 5.6 MB/s eta 0:00:00
     -------------------------------------- 118.1/118.1 kB 6.7 MB/s eta 0:00:00
     ------------------------------------- 506.2/506.2 kB 10.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import os
import cv2
import csv
import numpy as np
from rosbags.rosbag2 import Reader
from rosbags.typesys import stores
from rosbags.typesys.stores import Stores

# Absolute base path pointing to your Windows Desktop
desktop_path = r"C:\Users\stratklar\Documents\Markus\ASA\Indien\ASA_LIMO_data\my_all_sensor_data_test20"

# Defining the 3 distinct output folders directly on the desktop
color_output_dir = os.path.join(desktop_path, "extracted_images")
depth_output_dir = os.path.join(desktop_path, "extracted_depth")
lidar_output_dir = os.path.join(desktop_path, "extracted_lidar")
imu_output_dir = os.path.join(desktop_path, "extracted_imu")
odom_output_dir = os.path.join(desktop_path, "extracted_odom")

# Create the 3 individual directories automatically
os.makedirs(color_output_dir, exist_ok=True)
os.makedirs(depth_output_dir, exist_ok=True)
os.makedirs(lidar_output_dir, exist_ok=True)
os.makedirs(imu_output_dir, exist_ok=True)
os.makedirs(odom_output_dir, exist_ok=True)

# Define the absolute path for the final LiDAR CSV file inside its own folder
lidar_output_csv = os.path.join(lidar_output_dir, "lidar_data_v2.csv")
imu_output_csv = os.path.join(imu_output_dir, "imu_data_v3.csv")
odom_output_csv = os.path.join(odom_output_dir, "odom_data_v3.csv")

print("Targeting Desktop Rosbag data...")
print("Starting comprehensive data extraction into 3 separate folders...")

# Fetch the exact ROS2 Foxy message profiles used by the LIMO PRO
typestore = stores.get_typestore(Stores.ROS2_FOXY)

color_count = 0
depth_count = 0
lidar_count = 0
imu_count = 0
odom_count = 0

# Open the dedicated CSV file inside the LiDAR directory
with open(lidar_output_csv, mode='w', newline='') as lidar_file:
    csv_writer_lidar = csv.writer(lidar_file)
    #csv_writer_lidar.writerow(["Timestamp", "Angle_Min", "Angle_Max", "Angle_Increment", "Ranges_Array"])
    csv_writer_lidar.writerow(["Timestamp", "observation_index", "beam_index", "range"])

    with open(imu_output_csv, mode='w', newline='') as imu_file:
        csv_writer_imu = csv.writer(imu_file)
        #csv_writer_imu.writerow(["Timestamp", "qx", "qy", "qz", "qw", "ax", "ay", "az", "gx", "gy", "gz"])
        csv_writer_imu.writerow(["Timestamp", "index", "qx", "qy", "qz", "qw", "ax", "ay", "az", "gx", "gy", "gz", "cov_qxqx", "cov_qxqy", "cov_qxqz", "cov_qyqx", "cov_qyqy", "cov_qyqz", "cov_qzqx", "cov_qzqy", "cov_qzqz", "cov_axax", "cov_axay", "cov_axaz", "cov_ayax", "cov_ayay", "cov_ayaz", "cov_azax", "cov_azay", "cov_azaz", "cov_gxgx", "cov_gxgy", "cov_gxgz", "cov_gygx", "cov_gygy", "cov_gygz", "cov_gzgx", "cov_gzgy", "cov_gzgz"])

        with open(odom_output_csv, mode='w', newline='') as odom_file:
            csv_writer_odom = csv.writer(odom_file)
            columns_pose = [f"cov_pose{i}{j}" for i in range(6) for j in range(6)]
            columns_twist = [f"cov_twist{i}{j}" for i in range(6) for j in range(6)]

            csv_writer_odom.writerow(["Timestamp", "index", "x", "y", "z", "qx", "qy", "qz", "qw", "vx", "vy", "vz", "wx", "wy", "wz", *columns_pose, *columns_twist])
            
            # Initialize the binary reader directly over the desktop workspace
            with Reader(desktop_path) as reader:
                for connection, timestamp, rawdata in reader.messages():
                    
                    # --- 1. EXTRACT COLOR IMAGES ---
                    if connection.topic == '/camera/color/image_raw':
                        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                        
                        # Reconstruct image matrix from raw byte array
                        img = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, 3)
                        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
                        
                        # Save frame directly to the color folder
                        cv2.imwrite(os.path.join(color_output_dir, f"frame_{color_count:04d}.png"), img)
                        color_count += 1
                    
                    # --- 2. EXTRACT DEPTH CAMERA MAPS ---
                    elif connection.topic == '/camera/depth/image_raw':
                        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                        
                        # Unpack the 16-bit unsigned depth data (millimeter distances)
                        depth_img = np.frombuffer(msg.data, dtype=np.uint16).reshape(msg.height, msg.width)
                        
                        # Normalize 16-bit data to an 8-bit array (0-255) so human eyes can view it
                        visible_depth = cv2.normalize(depth_img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
                        
                        # Apply a high-contrast colormap (thermal style) for visual clarity
                        color_mapped_depth = cv2.applyColorMap(visible_depth, cv2.COLORMAP_JET)
                        
                        # Save depth image directly to the depth folder
                        cv2.imwrite(os.path.join(depth_output_dir, f"depth_{depth_count:04d}.png"), color_mapped_depth)
                        depth_count += 1
                    
                    # --- 3. EXTRACT LIDAR LASER SCANS ---
                    elif connection.topic == '/scan':
                        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                        
                        # Compress the float array of distance ranges into a clean string
                        ranges_str = ",".join([f"{r:.3f}" for r in msg.ranges])
                        
                        # Write row matching chronological timestamp
                        #with open(lidar_output_csv, mode='w', newline='') as csv_file:
                        #    csv_writer_lidar = csv.writer(csv_file)
                        ranges_count = (msg.angle_max - msg.angle_min) / msg.angle_increment + 1
                        beam_index = 0
                        for r in msg.ranges:
                            #csv_writer_lidar.writerow([timestamp, msg.angle_min, msg.angle_max, msg.angle_increment, ranges_str])
                            csv_writer_lidar.writerow([timestamp, lidar_count, beam_index, r])
                            beam_index += 1
                        lidar_count += 1
                    
                    # --- 4. EXTRACT IMU DATA
                    elif connection.topic == '/imu':
                        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                        print(msg)
                        # Compress the float array of distance ranges into a clean string
                        
                        # Write row matching chronological timestamp
                        #with open(imu_output_csv, mode='w', newline='') as csv_file2:
                            #csv_writer_imu = csv.writer(csv_file2)

                        #covariance of orientation quaternion
                        q_cov = []
                        i = 0
                        for q in msg.orientation_covariance:
                            if i not in [4, 7, 8]:
                                q_cov.append(q)
                            i += 1

                        #covariance of linear acceleration
                        a_cov = []
                        i = 0
                        for a in msg.linear_acceleration_covariance:
                            if i not in [6, 8, 9]:
                                a_cov.append(q)
                            i += 1

                        #covariance of angular velocity
                        g_cov = []
                        i = 0
                        for g in msg.angular_velocity_covariance:
                            if i not in [6, 8, 9]:
                                g_cov.append(q)
                            i += 1

                        csv_writer_imu.writerow([timestamp, imu_count, msg.orientation.x, msg.orientation.y, msg.orientation.z, msg.orientation.w, msg.linear_acceleration.x, msg.linear_acceleration.y, msg.linear_acceleration.z, msg.angular_velocity.x, msg.angular_velocity.y, msg.angular_velocity.z, *q_cov, *a_cov, *g_cov])
                        imu_count += 1

                    # --- 5. EXTRACT ODOM DATA
                    elif connection.topic == '/odom':
                        msg = typestore.deserialize_cdr(rawdata, connection.msgtype)
                        print(msg)

                        #cov_matrix = "[" + ",".join(["(" + ",".join([{msg.pose.covariance[7*j + i]} for i in range(7)]) + ")" for j in range(7)]) + "]"
                        # Write row matching chronological timestamp
                        #with open(odom_output_csv, mode='w', newline='') as csv_file3:
                        #csv_writer_odom= csv.writer(csv_file3)
                        csv_writer_odom.writerow([timestamp, odom_count, msg.pose.pose.position.x, msg.pose.pose.position.y, msg.pose.pose.position.z, msg.pose.pose.orientation.x, msg.pose.pose.orientation.y, msg.pose.pose.orientation.z, msg.pose.pose.orientation.w, msg.twist.twist.linear.x, msg.twist.twist.linear.y, msg.twist.twist.linear.z, msg.twist.twist.angular.x, msg.twist.twist.angular.y, msg.twist.twist.angular.z, *msg.pose.covariance, *msg.twist.covariance])
                        odom_count += 1

print("\n--- ALL DATA UNPACKED & SORTED SUCCESSFULLY ---")
print(f"Folder 1 [Color Camera]: Unpacked {color_count} images to: {color_output_dir}")
print(f"Folder 2 [Depth Camera]: Unpacked {depth_count} depth maps to: {depth_output_dir}")
print(f"Folder 3 [LiDAR Sensor]: Logged {lidar_count} scans to: {lidar_output_csv}")
print(f"Folder 4 [IMU Sensor]: Logged {imu_count} data points to: {imu_output_csv}")
print(f"Folder 5 [ODOM]: Logged {odom_count} data points to: {odom_output_csv}")

Targeting Desktop Rosbag data...
Starting comprehensive data extraction into 3 separate folders...
nav_msgs__msg__Odometry(header=std_msgs__msg__Header(stamp=builtin_interfaces__msg__Time(sec=1787659683, nanosec=979635104, __msgtype__='builtin_interfaces/msg/Time'), frame_id='odom', __msgtype__='std_msgs/msg/Header'), child_frame_id='base_link', pose=geometry_msgs__msg__PoseWithCovariance(pose=geometry_msgs__msg__Pose(position=geometry_msgs__msg__Point(x=25.635053982750822, y=-23.362326453170006, z=0.0, __msgtype__='geometry_msgs/msg/Point'), orientation=geometry_msgs__msg__Quaternion(x=0.0, y=0.0, z=-0.21763222269565075, w=0.9760308476910711, __msgtype__='geometry_msgs/msg/Quaternion'), __msgtype__='geometry_msgs/msg/Pose'), covariance=array([0.1, 0. , 0. , 0. , 0. , 0. , 0. , 0.1, 0. , 0. , 0. , 0. , 0. ,
       0. , 0.1, 0. , 0. , 0. , 0. , 0. , 0. , 1. , 0. , 0. , 0. , 0. ,
       0. , 0. , 1. , 0. , 0. , 0. , 0. , 0. , 0. , 1. ]), __msgtype__='geometry_msgs/msg/PoseWithCovariance'